### Importando Bibliotecas

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [12]:
df = pd.read_excel('Base_Operacoes.xlsx', sheet_name='Base_Operacoes')

In [ ]:
limite = 5_000_000

acima_sem_dupla = (df['Valor_R$'] > limite) & (df['Dupla_Aprovacao'] != 'Sim')
#abaixo_com_dupla = (df['Valor_R$'] < limite) & (df['Dupla_Aprovacao'] == 'Sim')

regra1_atendida = (
    ((df['Valor_R$'] > limite) & (df['Dupla_Aprovacao'] == 'Sim'))
    #| ((df['Valor_R$'] < limite) & (df['Dupla_Aprovacao'] == 'Não'))
)
df['Regra1'] = np.where(regra1_atendida, 'Sim', 'Não')

df['Regra1_Descricao'] = np.select(
    [acima_sem_dupla],
    ['Valor acima de R$ 5 milhões sem dupla aprovação'],
    default='Sem divergência identificada'
)
df

,ID_Operacao,Data_Operacao,ID_Cliente,ID_Operador,Produto,Valor_R$,Taxa_%_a.a.,Prazo_Dias,Data_Vencimento,Data_Liquidacao,Status,Dupla_Aprovacao,Data_Aprovacao,Canal,Praca,Regra1,Regra1_Descricao
0,OPR-2026-000001,2026-08-09,CLI-1101,OP-027,Debênture,7508250.96,13.01,720,2028-07-29,NaT,Cancelada,Sim,2026-08-09,Telefone,Curitiba,Sim,Sem divergência identificada
1,OPR-2026-000002,2026-09-08,CLI-1050,OP-032,CDB,1817329.52,11.83,90,2026-12-07,2026-09-08,Liquidada,Não,NaT,Telefone,Porto Alegre,Sim,Sem divergência identificada
2,OPR-2026-000003,2026-08-25,CLI-1529,OP-037,NTN-B,4768551.43,13.20,30,2026-09-24,2026-08-27,Liquidada,Sim,2026-08-25,Chat,Porto Alegre,Não,Valor abaixo de R$ 5 milhões com dupla aprovação
3,OPR-2026-000004,2026-08-23,CLI-1522,OP-004,LFT,1280409.26,12.03,360,2027-08-18,2026-08-23,Liquidada,Sim,2026-08-23,Chat,São Paulo,Não,Valor abaixo de R$ 5 milhões com dupla aprovação
4,OPR-2026-000005,2026-08-10,CLI-1280,OP-021,Debênture,5115043.49,13.39,180,2027-02-06,2026-08-10,Liquidada,Sim,2026-08-10,Chat,São Paulo,Sim,Sem divergência identificada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,OPR-2026-049996,2026-09-15,CLI-1399,OP-015,Debênture,6350074.74,11.71,360,2027-09-10,2026-09-15,Liquidada,Sim,2026-09-15,Telefone,Rio de Janeiro,Sim,Sem divergência identificada
49996,OPR-2026-049997,2026-09-25,CLI-1403,OP-040,LTN,4881958.54,10.76,60,2026-11-24,2026-09-25,Liquidada,Sim,2026-09-25,Chat,Curitiba,Não,Valor abaixo de R$ 5 milhões com dupla aprovação
49997,OPR-2026-049998,2026-09-11,CLI-1109,OP-035,LTN,6056019.16,14.52,90,2026-12-10,2026-09-12,Liquidada,Sim,2026-09-11,Telefone,Curitiba,Sim,Sem divergência identificada
49998,OPR-2026-049999,2026-09-16,CLI-1225,OP-021,Debênture,8011643.66,10.74,720,2028-09-05,NaT,Cancelada,Sim,2026-09-16,Telefone,Belo Horizonte,Sim,Sem divergência identificada


In [14]:
df['Data_validacao_vencimento'] = (
    df['Data_Operacao'] + pd.to_timedelta(df['Prazo_Dias'], unit='D')
)

df['Flag_Diferenca_Vencimento'] = (
    df['Data_validacao_vencimento'] == df['Data_Vencimento']
)

df[['Data_validacao_vencimento', 'Data_Vencimento', 'Flag_Diferenca_Vencimento']]

,Data_validacao_vencimento,Data_Vencimento,Flag_Diferenca_Vencimento
0,2028-07-29,2028-07-29,True
1,2026-12-07,2026-12-07,True
2,2026-09-24,2026-09-24,True
3,2027-08-18,2027-08-18,True
4,2027-02-06,2027-02-06,True
...,...,...,...
49995,2027-09-10,2027-09-10,True
49996,2026-11-24,2026-11-24,True
49997,2026-12-10,2026-12-10,True
49998,2028-09-05,2028-09-05,True


In [15]:
liquidada_com_data = (
    df['Status'].isin(['Liquidada', 'Concluída'])
    & df['Data_Liquidacao'].notna()
)
cancelada_ou_pendente_sem_data = (
    df['Status'].isin(['Cancelada', 'Pendente'])
    & df['Data_Liquidacao'].isna()
)

df['Verificacao_liquidada'] = (
    liquidada_com_data | cancelada_ou_pendente_sem_data
)

status = df['Status']
data_liquidacao = df['Data_Liquidacao']

df['Verificacao_Liquidada_Descricao'] = np.select(
    [
        df['Verificacao_liquidada'],
        (status == 'Liquidada') & data_liquidacao.isna(),
        (status == 'Concluída') & data_liquidacao.isna(),
        (status == 'Cancelada') & data_liquidacao.notna(),
        (status == 'Pendente') & data_liquidacao.notna()
    ],
    [
        'Normal',
        'Liquidada sem data',
        'Concluída sem data',
        'Cancelada com data',
        'Pendente com data'
    ],
    default='Status não previsto'
)

df[['Status', 'Data_Liquidacao', 'Verificacao_liquidada', 'Verificacao_Liquidada_Descricao']]

,Status,Data_Liquidacao,Verificacao_liquidada,Verificacao_Liquidada_Descricao
0,Cancelada,NaT,True,Normal
1,Liquidada,2026-09-08,True,Normal
2,Liquidada,2026-08-27,True,Normal
3,Liquidada,2026-08-23,True,Normal
4,Liquidada,2026-08-10,True,Normal
...,...,...,...,...
49995,Liquidada,2026-09-15,True,Normal
49996,Liquidada,2026-09-25,True,Normal
49997,Liquidada,2026-09-12,True,Normal
49998,Cancelada,NaT,True,Normal


In [17]:
df['Verificacao_Liquidada_Descricao'].value_counts(), df['Verificacao_Liquidada_Descricao'].value_counts(normalize=True)

(Verificacao_Liquidada_Descricao
 Normal                49976
 Cancelada com data       10
 Liquidada sem data       10
 Pendente com data         2
 Concluída sem data        2
 Name: count, dtype: int64,
 Verificacao_Liquidada_Descricao
 Normal                0.99952
 Cancelada com data    0.00020
 Liquidada sem data    0.00020
 Pendente com data     0.00004
 Concluída sem data    0.00004
 Name: proportion, dtype: float64)

In [20]:
data_limite_d1 = df['Data_Operacao'] + pd.Timedelta(days=1)
status_com_liquidacao = df['Status'].isin(['Liquidada', 'Concluída'])
liquidacao_com_data = df['Data_Liquidacao'].notna()

no_prazo_d1 = (
    status_com_liquidacao
    & liquidacao_com_data
    & df['Data_Liquidacao'].between(
        df['Data_Operacao'], data_limite_d1, inclusive='both'
    )
)
acima_d1 = (
    status_com_liquidacao
    & liquidacao_com_data
    & (df['Data_Liquidacao'] > data_limite_d1)
)
abaixo_data_operacao = (
    status_com_liquidacao
    & liquidacao_com_data
    & (df['Data_Liquidacao'] < df['Data_Operacao'])
)

df['Verificacao_D1'] = np.select(
    [no_prazo_d1, acima_d1, abaixo_data_operacao],
    ['ok', 'Acima', 'Abaixo'],
    default='Fora da Regra'
)

df[['Status', 'Data_Operacao', 'Data_Liquidacao', 'Verificacao_D1']]

,Status,Data_Operacao,Data_Liquidacao,Verificacao_D1
0,Cancelada,2026-08-09,NaT,Fora da Regra
1,Liquidada,2026-09-08,2026-09-08,ok
2,Liquidada,2026-08-25,2026-08-27,Acima
3,Liquidada,2026-08-23,2026-08-23,ok
4,Liquidada,2026-08-10,2026-08-10,ok
...,...,...,...,...
49995,Liquidada,2026-09-15,2026-09-15,ok
49996,Liquidada,2026-09-25,2026-09-25,ok
49997,Liquidada,2026-09-11,2026-09-12,ok
49998,Cancelada,2026-09-16,NaT,Fora da Regra


In [21]:
df['Verificacao_D1'].value_counts(), df['Verificacao_D1'].value_counts(normalize=True)

(Verificacao_D1
 ok               44490
 Fora da Regra     4535
 Acima              958
 Abaixo              17
 Name: count, dtype: int64,
 Verificacao_D1
 ok               0.88980
 Fora da Regra    0.09070
 Acima            0.01916
 Abaixo           0.00034
 Name: proportion, dtype: float64)

In [25]:
operadores_validos = {f'OP-{numero:03d}' for numero in range(1, 41)}
df['Verificacao_Operadores'] = df['ID_Operador'].isin(operadores_validos)

df[['ID_Operador', 'Verificacao_Operadores']]

,ID_Operador,Verificacao_Operadores
0,OP-027,True
1,OP-032,True
2,OP-037,True
3,OP-004,True
4,OP-021,True
...,...,...
49995,OP-015,True
49996,OP-040,True
49997,OP-035,True
49998,OP-021,True


In [26]:
df['Verificacao_Operadores'].value_counts(), df['Verificacao_Operadores'].value_counts(normalize=True)

(Verificacao_Operadores
 True     49990
 False       10
 Name: count, dtype: int64,
 Verificacao_Operadores
 True     0.9998
 False    0.0002
 Name: proportion, dtype: float64)

In [28]:
df[df['Verificacao_Operadores'] == False]

,ID_Operacao,Data_Operacao,ID_Cliente,ID_Operador,Produto,Valor_R$,Taxa_%_a.a.,Prazo_Dias,Data_Vencimento,Data_Liquidacao,...,Canal,Praca,Regra1,Regra1_Descricao,Data_validacao_vencimento,Flag_Diferenca_Vencimento,Verificacao_liquidada,Verificacao_Liquidada_Descricao,Verificacao_D1,Verificacao_Operadores
1434,OPR-2026-001435,2026-09-21,CLI-1541,OP-999,LTN,2239480.06,10.62,60,2026-11-20,2026-09-21,...,Chat,Curitiba,Sim,Sem divergência identificada,2026-11-20,True,True,Normal,ok,False
6374,OPR-2026-006375,2026-09-22,CLI-1016,OP-999,CDB,4261560.42,13.06,60,2026-11-21,2026-09-22,...,Telefone,Rio de Janeiro,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-11-21,True,True,Normal,ok,False
11314,OPR-2026-011315,2026-08-02,CLI-1523,OP-999,CDB,4779563.71,9.77,60,2026-10-01,2026-08-02,...,Telefone,São Paulo,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-10-01,True,True,Normal,ok,False
16254,OPR-2026-016255,2026-09-14,CLI-1427,OP-999,Debênture,8295174.78,12.92,30,2026-10-14,2026-09-14,...,E-mail,Belo Horizonte,Sim,Sem divergência identificada,2026-10-14,True,True,Normal,ok,False
21194,OPR-2026-021195,2026-08-10,CLI-1585,OP-999,NTN-B,1411822.58,13.50,180,2027-02-06,2026-08-10,...,E-mail,São Paulo,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2027-02-06,True,True,Normal,ok,False
26134,OPR-2026-026135,2026-08-30,CLI-1020,OP-999,NTN-B,2375451.48,13.66,30,2026-09-29,2026-08-30,...,E-mail,Porto Alegre,Sim,Sem divergência identificada,2026-09-29,True,True,Normal,ok,False
31074,OPR-2026-031075,2026-08-19,CLI-1517,OP-999,CDB,3333283.42,9.96,720,2028-08-08,2026-08-19,...,Telefone,Rio de Janeiro,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2028-08-08,True,True,Normal,ok,False
36014,OPR-2026-036015,2026-08-06,CLI-1038,OP-999,LFT,1891393.96,11.62,720,2028-07-26,NaT,...,Telefone,Porto Alegre,Sim,Sem divergência identificada,2028-07-26,True,True,Normal,Fora da Regra,False
40954,OPR-2026-040955,2026-08-10,CLI-1563,OP-999,CDB,3817425.30,8.47,30,2026-09-09,2026-08-10,...,Chat,Porto Alegre,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-09-09,True,True,Normal,ok,False
45894,OPR-2026-045895,2026-08-27,CLI-1571,OP-999,LTN,1702970.77,9.35,360,2027-08-22,2026-08-27,...,Chat,Belo Horizonte,Sim,Sem divergência identificada,2027-08-22,True,True,Normal,ok,False


In [29]:
taxa = df['Taxa_%_a.a.']

df['Verificacao_Taxa'] = taxa.between(8, 16, inclusive='both')
df['Verificacao_Taxa_Descricao'] = np.select(
    [taxa < 8, taxa > 16, taxa.isna()],
    [
        'Abaixo do intervalo de 8% a 16%',
        'Acima do intervalo de 8% a 16%',
        'Taxa ausente'
    ],
    default='Dentro do intervalo de 8% a 16%'
)

df[['Taxa_%_a.a.', 'Verificacao_Taxa', 'Verificacao_Taxa_Descricao']]

,Taxa_%_a.a.,Verificacao_Taxa,Verificacao_Taxa_Descricao
0,13.01,True,Dentro do intervalo de 8% a 16%
1,11.83,True,Dentro do intervalo de 8% a 16%
2,13.20,True,Dentro do intervalo de 8% a 16%
3,12.03,True,Dentro do intervalo de 8% a 16%
4,13.39,True,Dentro do intervalo de 8% a 16%
...,...,...,...
49995,11.71,True,Dentro do intervalo de 8% a 16%
49996,10.76,True,Dentro do intervalo de 8% a 16%
49997,14.52,True,Dentro do intervalo de 8% a 16%
49998,10.74,True,Dentro do intervalo de 8% a 16%


In [30]:
df['Verificacao_Taxa'].value_counts(), df['Verificacao_Taxa'].value_counts(normalize=True)

(Verificacao_Taxa
 True     49980
 False       20
 Name: count, dtype: int64,
 Verificacao_Taxa
 True     0.9996
 False    0.0004
 Name: proportion, dtype: float64)

In [31]:
df['Verificacao_Taxa_Descricao'].value_counts(), df['Verificacao_Taxa_Descricao'].value_counts(normalize=True)

(Verificacao_Taxa_Descricao
 Dentro do intervalo de 8% a 16%    49980
 Acima do intervalo de 8% a 16%        10
 Abaixo do intervalo de 8% a 16%       10
 Name: count, dtype: int64,
 Verificacao_Taxa_Descricao
 Dentro do intervalo de 8% a 16%    0.9996
 Acima do intervalo de 8% a 16%     0.0002
 Abaixo do intervalo de 8% a 16%    0.0002
 Name: proportion, dtype: float64)

In [32]:
df[df['Status']=='Pendente']['Data_Operacao'].describe()

count                          2987
mean     2026-08-31 05:21:04.278540
min             2026-08-01 00:00:00
25%             2026-08-16 00:00:00
50%             2026-08-31 00:00:00
75%             2026-09-15 00:00:00
max             2027-02-15 00:00:00
Name: Data_Operacao, dtype: object

In [34]:
data_corte_pendencia = pd.Timestamp('2026-09-30')
data_vencimento_pendencia = (
    df['Data_Operacao'] + pd.to_timedelta(df['Prazo_Dias'], unit='D')
)
pendencia_ate_corte = (
    (df['Status'] == 'Pendente')
    & df['Data_Operacao'].notna()
    & (df['Data_Operacao'] <= data_corte_pendencia)
)

pendencia_dentro_prazo = (
    pendencia_ate_corte
    & (data_vencimento_pendencia > data_corte_pendencia)
)
pendencia_fora_prazo = (
    pendencia_ate_corte
    & (data_vencimento_pendencia <= data_corte_pendencia)
)

df['Verificacao_Pendencia_30_09'] = np.select(
    [pendencia_dentro_prazo, pendencia_fora_prazo],
    ['Dentro do Prazo', 'Fora do Prazo'],
    default='Fora da Regra'
)

df[['Status', 'Data_Operacao', 'Prazo_Dias', 'Verificacao_Pendencia_30_09']]

,Status,Data_Operacao,Prazo_Dias,Verificacao_Pendencia_30_09
0,Cancelada,2026-08-09,720,Fora da Regra
1,Liquidada,2026-09-08,90,Fora da Regra
2,Liquidada,2026-08-25,30,Fora da Regra
3,Liquidada,2026-08-23,360,Fora da Regra
4,Liquidada,2026-08-10,180,Fora da Regra
...,...,...,...,...
49995,Liquidada,2026-09-15,360,Fora da Regra
49996,Liquidada,2026-09-25,60,Fora da Regra
49997,Liquidada,2026-09-11,90,Fora da Regra
49998,Cancelada,2026-09-16,720,Fora da Regra


In [35]:
df['Verificacao_Pendencia_30_09'].value_counts(), df['Verificacao_Pendencia_30_09'].value_counts(normalize=True)

(Verificacao_Pendencia_30_09
 Fora da Regra      47014
 Dentro do Prazo     2709
 Fora do Prazo        277
 Name: count, dtype: int64,
 Verificacao_Pendencia_30_09
 Fora da Regra      0.94028
 Dentro do Prazo    0.05418
 Fora do Prazo      0.00554
 Name: proportion, dtype: float64)

In [38]:
df

,ID_Operacao,Data_Operacao,ID_Cliente,ID_Operador,Produto,Valor_R$,Taxa_%_a.a.,Prazo_Dias,Data_Vencimento,Data_Liquidacao,Status,Dupla_Aprovacao,Data_Aprovacao,Canal,Praca,Regra1,Regra1_Descricao,Data_validacao_vencimento,Flag_Diferenca_Vencimento,Verificacao_liquidada,Verificacao_Liquidada_Descricao,Verificacao_D1,Verificacao_Operadores,Verificacao_Taxa,Verificacao_Taxa_Descricao,Verificacao_Pendencia_30_09
0,OPR-2026-000001,2026-08-09,CLI-1101,OP-027,Debênture,7508250.96,13.01,720,2028-07-29,NaT,Cancelada,Sim,2026-08-09,Telefone,Curitiba,Sim,Sem divergência identificada,2028-07-29,True,True,Normal,Fora da Regra,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
1,OPR-2026-000002,2026-09-08,CLI-1050,OP-032,CDB,1817329.52,11.83,90,2026-12-07,2026-09-08,Liquidada,Não,NaT,Telefone,Porto Alegre,Sim,Sem divergência identificada,2026-12-07,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
2,OPR-2026-000003,2026-08-25,CLI-1529,OP-037,NTN-B,4768551.43,13.20,30,2026-09-24,2026-08-27,Liquidada,Sim,2026-08-25,Chat,Porto Alegre,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-09-24,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
3,OPR-2026-000004,2026-08-23,CLI-1522,OP-004,LFT,1280409.26,12.03,360,2027-08-18,2026-08-23,Liquidada,Sim,2026-08-23,Chat,São Paulo,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2027-08-18,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
4,OPR-2026-000005,2026-08-10,CLI-1280,OP-021,Debênture,5115043.49,13.39,180,2027-02-06,2026-08-10,Liquidada,Sim,2026-08-10,Chat,São Paulo,Sim,Sem divergência identificada,2027-02-06,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,OPR-2026-049996,2026-09-15,CLI-1399,OP-015,Debênture,6350074.74,11.71,360,2027-09-10,2026-09-15,Liquidada,Sim,2026-09-15,Telefone,Rio de Janeiro,Sim,Sem divergência identificada,2027-09-10,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
49996,OPR-2026-049997,2026-09-25,CLI-1403,OP-040,LTN,4881958.54,10.76,60,2026-11-24,2026-09-25,Liquidada,Sim,2026-09-25,Chat,Curitiba,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-11-24,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
49997,OPR-2026-049998,2026-09-11,CLI-1109,OP-035,LTN,6056019.16,14.52,90,2026-12-10,2026-09-12,Liquidada,Sim,2026-09-11,Telefone,Curitiba,Sim,Sem divergência identificada,2026-12-10,True,True,Normal,ok,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra
49998,OPR-2026-049999,2026-09-16,CLI-1225,OP-021,Debênture,8011643.66,10.74,720,2028-09-05,NaT,Cancelada,Sim,2026-09-16,Telefone,Belo Horizonte,Sim,Sem divergência identificada,2028-09-05,True,True,Normal,Fora da Regra,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra


In [ ]:
x = df[df['Status'] != 'Cancelada'].groupby('ID_Operador').agg({'Valor_R$': 'sum', 'ID_Operacao': 'count'}).reset_index()


def validacao_erro(row):
    erro = (
        (row['Regra1'] == 'Não')
        or not row['Flag_Diferenca_Vencimento']
        or not row['Verificacao_liquidada']
        or row['Verificacao_D1'] in ('Acima', 'Abaixo')
        or not row['Verificacao_Operadores']
        or not row['Verificacao_Taxa']
        or row['Verificacao_Pendencia_30_09'] == 'Fora do Prazo'
    )
    return 'Erro' if erro else 'Sem Erro'


y = df.loc[df['Status'] != 'Cancelada'].copy()
y['Status_Validacao'] = y.apply(validacao_erro, axis=1)

z = (
    y.loc[y['Status_Validacao'] == 'Erro']
    .groupby('ID_Operador')
    .agg({'ID_Operacao': 'count', 'Valor_R$': 'sum'})
    .rename(columns={
        'ID_Operacao': 'ID_Operacao_erro',
        'Valor_R$': 'Valor_R$_erro'
    })
    .reset_index()
)
z

,ID_Operador,ID_Operacao_erro,Valor_R$_erro
0,OP-001,557,1.504272e+09
1,OP-002,508,1.341774e+09
2,OP-003,522,1.378488e+09
3,OP-004,509,1.314497e+09
4,OP-005,488,1.324258e+09
5,OP-006,530,1.409415e+09
6,OP-007,512,1.380868e+09
7,OP-008,558,1.448204e+09
8,OP-009,513,1.308241e+09
9,OP-010,499,1.339329e+09


In [54]:
x = x.merge(z, on='ID_Operador', how='left').fillna(0)

In [56]:
x['Proporção_erro'] = x['ID_Operacao_erro'] / x['ID_Operacao']

In [ ]:
x.sort_values(by='Proporção_erro', ascending=False).head(10)

,ID_Operador,Valor_R$,ID_Operacao,ID_Operacao_erro,Valor_R$_erro,Proporção_erro
40,OP-999,3.410813e+07,10,10,3.410813e+07,1.000000
23,OP-024,6.153597e+09,1286,596,1.640851e+09,0.463453
35,OP-036,5.460676e+09,1163,531,1.432215e+09,0.456578
12,OP-013,5.632270e+09,1225,559,1.462270e+09,0.456327
24,OP-025,5.494359e+09,1205,544,1.344328e+09,0.451452
5,OP-006,5.608217e+09,1183,532,1.411452e+09,0.449704
11,OP-012,5.551953e+09,1160,520,1.375851e+09,0.448276
38,OP-039,6.022620e+09,1234,552,1.547156e+09,0.447326
33,OP-034,5.920861e+09,1240,554,1.434817e+09,0.446774
7,OP-008,6.282851e+09,1275,568,1.502697e+09,0.445490


In [50]:
y[y['Status_Validacao']=='Erro']['Produto'].value_counts()

Produto
LTN          4242
CDB          4219
Debênture    4207
NTN-B        4186
LFT          4135
Name: count, dtype: int64

In [39]:
df.to_excel('Base_Operacoes_Tratada.xlsx', index=False)

In [62]:
def operacoes_dupla_aprovacao_com_erro(df: pd.DataFrame) -> pd.DataFrame:
    violacoes = pd.DataFrame({
        'Flag_Diferenca_Vencimento': ~df['Flag_Diferenca_Vencimento'].astype(bool),
        'Verificacao_liquidada':     ~df['Verificacao_liquidada'].astype(bool),
        'Verificacao_D1':            df['Verificacao_D1'].isin(['Acima', 'Abaixo']),
        'Verificacao_Operadores':    ~df['Verificacao_Operadores'].astype(bool),
        'Verificacao_Taxa':          ~df['Verificacao_Taxa'].astype(bool),
    })

    filtro = (df['Dupla_Aprovacao'] == 'Sim') & violacoes.any(axis=1)

    resultado = df.loc[filtro].copy()
    v = violacoes.loc[filtro]
    resultado['Qtd_Regras_Quebradas'] = v.sum(axis=1)
    resultado['Regras_Quebradas'] = v.apply(lambda l: '; '.join(l.index[l]), axis=1)
    return resultado

# uso
erros = operacoes_dupla_aprovacao_com_erro(df)
erros

,ID_Operacao,Data_Operacao,ID_Cliente,ID_Operador,Produto,Valor_R$,Taxa_%_a.a.,Prazo_Dias,Data_Vencimento,Data_Liquidacao,Status,Dupla_Aprovacao,Data_Aprovacao,Canal,Praca,Regra1,Regra1_Descricao,Data_validacao_vencimento,Flag_Diferenca_Vencimento,Verificacao_liquidada,Verificacao_Liquidada_Descricao,Verificacao_D1,Verificacao_Operadores,Verificacao_Taxa,Verificacao_Taxa_Descricao,Verificacao_Pendencia_30_09,Dupla_Aprovacao_Realizada,Regras_Quebradas_1_a_6,Status_Validacao,Qtd_Regras_Quebradas,Regras_Quebradas
2,OPR-2026-000003,2026-08-25,CLI-1529,OP-037,NTN-B,4768551.43,13.20,30,2026-09-24,2026-08-27,Liquidada,Sim,2026-08-25,Chat,Porto Alegre,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-09-24,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,"Regra 1, Regra 4",Erro,1,Verificacao_D1
64,OPR-2026-000065,2026-08-17,CLI-1236,OP-036,CDB,1975180.68,13.82,360,2027-08-12,2026-08-19,Liquidada,Sim,2026-08-17,E-mail,Belo Horizonte,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2027-08-12,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,"Regra 1, Regra 4",Erro,1,Verificacao_D1
218,OPR-2026-000219,2026-08-18,CLI-1024,OP-016,LFT,8118153.13,15.27,180,2027-02-14,2026-08-20,Liquidada,Sim,2026-08-18,Telefone,Porto Alegre,Sim,Sem divergência identificada,2027-02-14,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,Regra 4,Erro,1,Verificacao_D1
244,OPR-2026-000245,2026-08-13,CLI-1054,OP-005,NTN-B,1456846.83,8.44,180,2027-02-09,2026-08-15,Liquidada,Sim,2026-08-13,E-mail,Belo Horizonte,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2027-02-09,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,"Regra 1, Regra 4",Erro,1,Verificacao_D1
389,OPR-2026-000390,2026-09-21,CLI-1575,OP-030,LFT,9196370.60,12.13,60,2026-11-20,2026-09-23,Liquidada,Sim,2026-09-21,E-mail,Belo Horizonte,Sim,Sem divergência identificada,2026-11-20,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,Regra 4,Erro,1,Verificacao_D1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49514,OPR-2026-049515,2026-09-08,CLI-1305,OP-021,CDB,4868453.66,13.31,360,2027-09-03,2026-09-10,Liquidada,Sim,2026-09-08,Telefone,Rio de Janeiro,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2027-09-03,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,"Regra 1, Regra 4",Erro,1,Verificacao_D1
49586,OPR-2026-049587,2026-09-29,CLI-1093,OP-009,Debênture,7591961.37,9.93,60,2026-11-28,2026-10-01,Liquidada,Sim,2026-09-29,Telefone,Porto Alegre,Sim,Sem divergência identificada,2026-11-28,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,Regra 4,Erro,1,Verificacao_D1
49680,OPR-2026-049681,2026-09-04,CLI-1419,OP-030,NTN-B,6570008.16,14.65,360,2027-08-30,2026-09-06,Liquidada,Sim,2026-09-04,E-mail,Porto Alegre,Sim,Sem divergência identificada,2027-08-30,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,Regra 4,Erro,1,Verificacao_D1
49806,OPR-2026-049807,2026-09-25,CLI-1267,OP-030,LTN,2750776.25,12.52,30,2026-10-25,2026-09-27,Liquidada,Sim,2026-09-25,Telefone,Rio de Janeiro,Não,Valor abaixo de R$ 5 milhões com dupla aprovação,2026-10-25,True,True,Normal,Acima,True,True,Dentro do intervalo de 8% a 16%,Fora da Regra,True,"Regra 1, Regra 4",Erro,1,Verificacao_D1


In [68]:

def filtrar_operacoes_com_quebra(df):

    base = df.copy()

    # Tratamento das datas
    col_datas = [
        'Data_Operacao',
        'Data_Vencimento',
        'Data_Liquidacao',
        'Data_Aprovacao'
    ]

    for col in col_datas:
        base[col] = pd.to_datetime(base[col], errors='coerce')

    valor = pd.to_numeric(base['Valor_R$'], errors='coerce')
    prazo = pd.to_numeric(base['Prazo_Dias'], errors='coerce')
    taxa = pd.to_numeric(base['Taxa_%_a.a.'], errors='coerce')

    status = base['Status'].astype('string').str.strip()

    # Datas calculadas
    vencimento_calculado = (
        base['Data_Operacao'] + pd.to_timedelta(prazo, unit='D')
    )

    # REGRA 1 - Dupla aprovação
    dupla = (
        base['Dupla_Aprovacao'].astype('string').str.strip().eq('Sim')
    )

    aprovacao_previa = (
        dupla
        & base['Data_Aprovacao'].notna()
        & base['Data_Operacao'].notna()
        & (base['Data_Aprovacao'] <= base['Data_Operacao'])
    )

    quebra_r1 = (
        ((valor > 5_000_000) & ~aprovacao_previa)
        | ((valor < 5_000_000) & dupla)
    )

    # REGRA 2 - Compatibilidade do vencimento
    quebra_r2 = ~(
        base['Data_Vencimento'].notna()
        & vencimento_calculado.notna()
        & (base['Data_Vencimento'] == vencimento_calculado)
    )

    # REGRA 3 - Coerência entre status e data de liquidação
    regra3_atendida = (
        (status.isin(['Liquidada', 'Concluída'])
         & base['Data_Liquidacao'].notna())
        |
        (status.isin(['Cancelada', 'Pendente'])
         & base['Data_Liquidacao'].isna())
    )

    quebra_r3 = ~regra3_atendida

    # REGRA 4 - Liquidação até D+1 (dias corridos)
    status_liquidado = status.isin(['Liquidada', 'Concluída'])

    quebra_r4 = (
        status_liquidado
        & base['Data_Liquidacao'].notna()
        & base['Data_Operacao'].notna()
        & (
            (base['Data_Liquidacao'] < base['Data_Operacao'])
            |
            (base['Data_Liquidacao'] >
             base['Data_Operacao'] + pd.Timedelta(days=1))
        )
    )

    # REGRA 5 - IDs únicos e operadores válidos
    operadores_validos = {
        f'OP-{numero:03d}' for numero in range(1, 41)
    }

    operador_valido = (
        base['ID_Operador'].astype('string').str.strip()
        .isin(operadores_validos)
    )

    id_operacao = base['ID_Operacao'].astype('string').str.strip()

    id_duplicado = id_operacao.duplicated(keep=False)

    id_ausente = id_operacao.isna() | id_operacao.eq('')

    quebra_r5 = ~operador_valido | id_duplicado | id_ausente

    # REGRA 6 - Taxas entre 8% e 16% a.a.
    quebra_r6 = ~taxa.between(8, 16, inclusive='both')

    # REGRA 7 - Pendências fora do prazo em 30/09/2026
    data_corte = pd.Timestamp('2026-09-30')

    pendencia_ate_corte = (
        (status == 'Pendente')
        & base['Data_Operacao'].notna()
        & (base['Data_Operacao'] <= data_corte)
    )

    quebra_r7 = (
        pendencia_ate_corte
        & vencimento_calculado.notna()
        & (vencimento_calculado <= data_corte)
    )

    # Consolidando as regras
    regras = pd.DataFrame({
        'Regra 1': quebra_r1,
        'Regra 2': quebra_r2,
        'Regra 3': quebra_r3,
        'Regra 4': quebra_r4,
        'Regra 5': quebra_r5,
        'Regra 6': quebra_r6,
        'Regra 7': quebra_r7
    }, index=base.index)

    # Selecionando operações com pelo menos uma quebra
    mascara = regras.any(axis=1)

    resultado = base.loc[mascara].copy()

    # Identificando quais regras foram quebradas
    resultado['Regras_Quebradas'] = regras.loc[mascara].apply(
        lambda linha: ', '.join(
            linha.index[linha.values]
        ),
        axis=1
    )

    # Quantidade de regras quebradas por operação
    resultado['Qtd_Regras_Quebradas'] = (
        regras.loc[mascara].sum(axis=1)
    )

    return resultado

operacoes_com_quebra = filtrar_operacoes_com_quebra(df)
operacoes_com_quebra['Valor_R$'].sum()

np.float64(58347304108.41)